# Geometry-V3 P1M0 posterior mechanism audit
One artifact-bound, identity-only diagnostic run. Outputs remain operational evidence with `science_denominator=0`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import datetime as dt
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
from google.colab import userdata

REPOSITORY = 'https://github.com/RICHAAARC/CEG-WM.git'
P1M0_RUNNER_EXACT = '73c218e24811d880c4e135e008cd4b8ddce9baa2'
RUNNER_RELATIVE_PATH = 'experiments/run_geometry_v3_qk_active_writer_p1m0_mechanism_audit.py'
P0_SOURCE_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V3/P0/Geometry-V3-P0-9b5085c805b6-20260828T122005Z')
P1_SOURCE_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V3/P1/Geometry-V3-P1-517ba73993f1-20260828T131759Z')
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V3/P1M0')
MAX_CONTROL_BYTES = 1024

hf_token = userdata.get('HF_TOKEN')
geometry_key = userdata.get('CEGWM_GEOMETRY_KEY')
if not isinstance(hf_token, str) or not hf_token.strip():
    raise RuntimeError('HF_TOKEN userdata is required')
if not isinstance(geometry_key, str) or not geometry_key.strip():
    raise RuntimeError('CEGWM_GEOMETRY_KEY userdata is required')

checkout = Path(tempfile.mkdtemp(prefix='cegwm-geometry-v3-p1m0-checkout-'))
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(checkout)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['git', 'checkout', '--detach', P1M0_RUNNER_EXACT], cwd=checkout, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
resolved = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=checkout, check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=checkout, check=True, capture_output=True, text=True).stdout
if resolved != P1M0_RUNNER_EXACT or dirty:
    raise RuntimeError('P1M0 detached checkout identity differs')
runner_path = checkout / RUNNER_RELATIVE_PATH
if not runner_path.is_file():
    raise RuntimeError('P1M0 runner is absent from the bound checkout')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

timestamp = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
drive_directory = DRIVE_ROOT / f'Geometry-V3-P1M0-{P1M0_RUNNER_EXACT[:12]}-{timestamp}'
if drive_directory.exists():
    raise FileExistsError('P1M0 Drive output already exists')
plan = {
    'expected_exact': P1M0_RUNNER_EXACT,
    'execution_exact': P1M0_RUNNER_EXACT,
    'p0_source_directory': str(P0_SOURCE_ROOT),
    'p1_source_directory': str(P1_SOURCE_ROOT),
    'output_directory': str(drive_directory),
}
plan_file = Path(tempfile.mkdtemp(prefix='cegwm-geometry-v3-p1m0-plan-')) / 'plan.json'
plan_file.write_text(json.dumps(plan, sort_keys=True, separators=(',', ':')), encoding='utf-8')
child_env = os.environ.copy()
child_env['HF_TOKEN'] = hf_token
child_env['CEGWM_GEOMETRY_KEY'] = geometry_key
hf_token = ''
geometry_key = ''
read_fd, write_fd = os.pipe()
try:
    child = subprocess.Popen(
        [sys.executable, str(runner_path), '--plan', str(plan_file), '--control-fd', str(write_fd)],
        cwd=checkout, env=child_env, pass_fds=(write_fd,),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    os.close(write_fd)
    write_fd = -1
    runner_rc = child.wait(timeout=7200)
    control_bytes = os.read(read_fd, MAX_CONTROL_BYTES + 1)
finally:
    child_env.pop('HF_TOKEN', None)
    child_env.pop('CEGWM_GEOMETRY_KEY', None)
    if write_fd >= 0:
        os.close(write_fd)
    os.close(read_fd)
if len(control_bytes) > MAX_CONTROL_BYTES:
    raise RuntimeError('P1M0 control receipt exceeds bound')
control = json.loads(control_bytes) if control_bytes else {}
terminal = {
    'runner_execution_identity': {'commit': P1M0_RUNNER_EXACT, 'clean': True},
    'source_p0_artifact_identity': {
        'execution_exact': '9b5085c805b6e3580fadc153598aac93fcc41eab',
        'run_id': 'geometry-v3-qk-p0-9b5085c805b6',
    },
    'source_p1_artifact_identity': {
        'execution_exact': '517ba73993f11f51ade27fee181814294fe53797',
        'run_id': 'geometry-v3-qk-p1-517ba73993f1',
    },
    'runner_rc': runner_rc,
    'drive_directory': str(drive_directory),
    'handoff_status': control.get('status'),
    'status': control.get('p1m0_status'),
    'run_id': control.get('run_id'),
    'artifact_status': control.get('artifact_status', 'unavailable'),
    'fixed_config_id': control.get('fixed_config_id'),
    'failure_point': control.get('failure_point'),
    'error_class': control.get('error_class'),
    'science_denominator': 0,
}
print('CEGWM_GEOMETRY_V3_QK_P1M0_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
if runner_rc != 0 or control.get('status') != 'success':
    raise RuntimeError('Geometry-V3 P1M0 runner failed; inspect bounded terminal fields')